In [1]:
import os

import matplotlib.pyplot as plt
import numpy as np

import sam3
from PIL import Image,ImageOps,ImageEnhance
from sam3 import build_sam3_image_model
from sam3.model.box_ops import box_xywh_to_cxcywh
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.visualization_utils import draw_box_on_image, normalize_bbox, plot_results

import torch
import cv2

torch.cuda.empty_cache()

# 1. Check if CUDA is available
print(f"Is CUDA available: {torch.cuda.is_available()}")

# 2. Get the name of the current GPU (if available)
if torch.cuda.is_available():
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")


/home/leo/sam3/sam3/model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/leo/miniforge3/envs/SAM3OpenPIV/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Is CUDA available: True
Current device: 0
Device name: NVIDIA GeForce RTX 3070 Laptop GPU


In [2]:
#torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Load the model
model = build_sam3_image_model()
processor = Sam3Processor(model,confidence_threshold=0.5)

Name1="Frame_00001.tif" #Change name for PIV image used to generate mask
InputPath=os.path.join(os.getcwd(),"input")
GenPath=os.path.join(os.getcwd(),"input",Name1)

# depth, height, width = img.shape
# print(f"Depth: {depth}")
# print(f"Height: {height}") 
# print(f"Width: {width}")  

# image = np.array(Image.open(GenPath).convert('RGB'))
# depth, height, width = image.shape
# print(f"Depth: {depth}")
# print(f"Height: {height}") 
# print(f"Width: {width}")  

# cv2.namedWindow("MainWindow", cv2.WINDOW_NORMAL)
# cv2.resizeWindow("MainWindow", 1000, 700)

# cv2.imshow('MainWindow', img)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

# Load an image
image = Image.open(GenPath).convert('RGB')
depth, height, width = np.array(image).shape
print(f"Depth: {depth}")
print(f"Height: {height}") 
print(f"Width: {width}")  

img_with_border = ImageOps.expand(image,border=300,fill='white')
EnhancerInstance = ImageEnhance.Brightness(img_with_border)
ImageInput=EnhancerInstance.enhance(1)
ImageInput.show()
# inference_state = processor.set_image(image)

# # Prompt the model with text
# output = processor.set_text_prompt(state=inference_state, prompt="black")

# # Get the masks, bounding boxes, and scores
# masks, boxes, scores = output["masks"], output["boxes"], output["scores"]

Depth: 2052
Height: 4603
Width: 3


In [3]:
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    inference_state = processor.set_image(img_with_border)
    output = processor.set_text_prompt(state=inference_state, prompt="Black Shape")
    masks, boxes, scores = output["masks"], output["boxes"], output["scores"]

if len(masks) == 0:
    print(f"No objects of the prompt detected in the image.")
else:
    print(f"Detected")
    
    
    masks_np = [mask.squeeze().cpu().numpy() for mask in masks]

    
    print(scores)
    mask_uint8 = masks_np[0].astype(np.uint8) * 255
    CroppedMask=mask_uint8[300:depth+300, 300:height+300]
    # height, width = CroppedMask.shape
    # print(f"Height: {height}") 
    # print(f"Width: {width}")      
    
    maskimage=Image.fromarray(CroppedMask)
    maskimage.show()
# cv2.namedWindow("2w", cv2.WINDOW_NORMAL)
    # cv2.resizeWindow("2w", 1000, 700)

    # cv2.imshow('2w', imMask)




Detected
tensor([0.9492], device='cuda:0', dtype=torch.bfloat16)


In [4]:
    Source = cv2.imread(GenPath)

    color_mask = cv2.cvtColor(CroppedMask, cv2.COLOR_GRAY2BGR)

    color_mask[CroppedMask == 255] = [255, 0, 0] # Blue BGR

    img = cv2.addWeighted(color_mask, 0.3, Source, 1, 0)

    cv2.namedWindow("2w", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("2w", 1000, 700)

    cv2.imshow('2w', img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [5]:
torch.cuda.empty_cache()